LIBRARIES

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

KNOWLEDGE BASE

In [ ]:
knowledge_base = [
    "To reset your password, click on the Forgot Password link on the login page.",
    "Billing invoices can be downloaded from the billing section of your account.",
    "If your account is locked, contact customer support for assistance.",
    "You can update your profile information from the account settings page.",
    "Subscription plans can be upgraded or downgraded anytime.",
    "If you cannot log in, ensure your username and password are correct.",
    "Refund requests are processed within five to seven business days.",
    "Two-factor authentication adds an extra layer of account security.",
    "Payment methods can be managed in the billing dashboard.",
    "Account deletion requests must be submitted through the support portal."
]

LOADING EMBEDDING MODEL

In [ ]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

GENERATING EMBEDDINGS

In [ ]:
embeddings = model.encode(knowledge_base)

print("\nEmbedding Matrix Shape:")
print(embeddings.shape)

BUILDING FAISS INDEX

In [ ]:
dimension = embeddings.shape[1] 

index = faiss.IndexFlatL2(dimension)

NORMALIZING EMBEDDINGS FOR COSINE SIMILARITY BEHAVIOUR

In [ ]:
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

ADDING VECTORS TO THE FAISS INDEX

In [ ]:
index.add(embeddings)

print("\nTotal Vectors Stored in FAISS:")
print(index.ntotal)

SEMANTIC SEARCH FUNCTION

In [ ]:
def semantic_search(query, k=3):
    query_embedding = model.encode([query]).astype("float32")

    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, k)

    print("\nTop Matches:")
    print("-" * 90)
    print(f"{'Rank':<6}{'Score':<15}{'Matched Sentence'}")
    print("-" * 90)

    for rank, (idx, score) in enumerate(
        zip(indices[0], distances[0]), start=1
    ):
        print(
            f"{rank:<6}{score:<15.4f}{knowledge_base[idx]}"
        )



TESTING QUERIES

In [ ]:
test_queries = [
    "How do I change my password?",
    "Where can I see my invoice?",
    "I am unable to access my account"
]

print("\n\n========== SAMPLE SEARCH RESULTS ==========")

for query in test_queries:
    print(f"\nQuery: {query}")
    semantic_search(query)


INTERACTIVE CLI

In [ ]:
print("\n\n========== INTERACTIVE SEARCH ==========")

while True:
    query = input("\nEnter your query (or type 'exit'): ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    semantic_search(query)

## Q1: What is the difference between IndexFlatL2 and IndexFlatIP in FAISS? When would you use each?

### IndexFlatL2

IndexFlatL2 uses Euclidean (L2) distance to measure similarity between vectors.

Formula:

L2 Distance = √Σ(xᵢ - yᵢ)²

Smaller distance indicates greater similarity.

Use Cases:
- When Euclidean distance is the desired similarity metric.
- Useful when vector magnitude is important.

---

### IndexFlatIP

IndexFlatIP uses Inner Product.

Formula:

x · y

Higher values indicate greater similarity.

Use Cases:
- Semantic search
- Retrieval-Augmented Generation (RAG)
- Recommendation systems

When embeddings are normalized, Inner Product behaves like Cosine Similarity.

## Q2: Why do we normalize embeddings before adding them to FAISS when we want cosine similarity?

Cosine similarity measures the angle between vectors rather than their magnitude.

Formula:

Cosine Similarity = (A · B) / (||A|| ||B||)

Normalization converts every vector to unit length.

After normalization:

||A|| = ||B|| = 1

Therefore:

Cosine Similarity = A · B

This allows FAISS to perform cosine-style searches efficiently.


## Q3: FAISS uses ANN (Approximate Nearest Neighbour) search. What does "approximate" mean here and why is it acceptable?

Approximate Nearest Neighbour (ANN) search tries to find vectors that are very close to the true nearest neighbours without checking every vector.

Advantages:

- Much faster search
- Lower memory usage
- Scales to millions or billions of vectors

Trade-Off:

- Results may not always be the exact nearest neighbour.
- Results are usually extremely close to the exact answer.

This trade-off is acceptable because retrieval systems prioritize speed while maintaining high accuracy.